# ⚛️ Quantum-Inspired Metaheuristic Route Optimization: Benchmark & Convergence Analysis

This notebook provides experimental benchmarking and mathematical analysis of **Quantum-Behaved Particle Swarm Optimization (QPSO)** compared against classical metaheuristics (**Simulated Annealing**, **Genetic Algorithms**, **Ant Colony Optimization**, and **Classical PSO**) for Capacitated Vehicle Routing with Time Windows (CVRPTW) under dynamic traffic congestion.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add backend to system path
sys.path.insert(0, os.path.abspath('..'))

from backend.app.algorithms import ALGORITHM_REGISTRY, qpso_solver
from backend.app.benchmarking.runner import benchmark_runner
from backend.app.core.traffic_sim import traffic_sim
from backend.app.core.graph_model import transportation_graph

## 1. Load Dataset & Inspect Stops

In [ ]:
df_stops = pd.read_csv('../data/demo_graphs/10_nodes_city.csv')
print(f"Loaded {len(df_stops)} nodes:")
df_stops.head(10)

## 2. Dynamic Traffic Congestion Profile $\theta(t)$

In [ ]:
hours = np.linspace(0, 24, 200)
multipliers = [traffic_sim.get_congestion_factor(h) for h in hours]

plt.figure(figsize=(10, 4))
plt.plot(hours, multipliers, color='#00f3ff', lw=2.5)
plt.axvspan(8.0, 10.0, color='red', alpha=0.15, label='Morning Rush (8-10 AM)')
plt.axvspan(17.0, 19.5, color='orange', alpha=0.15, label='Evening Rush (5-7:30 PM)')
plt.title('Time-of-Day Traffic Congestion Multiplier $\\theta(t)$')
plt.xlabel('Hour of Day')
plt.ylabel('Congestion Multiplier')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

## 3. Execute Multi-Algorithm Benchmark Suite

In [ ]:
start_node = {
    'name': df_stops.iloc[0]['name'],
    'coords': (df_stops.iloc[0]['lat'], df_stops.iloc[0]['lon'])
}
stops_data = [
    {
        'name': row['name'],
        'coords': (row['lat'], row['lon']),
        'demand': row.get('demand', 1.0),
        'window': (row.get('start_time', 9.0), row.get('end_time', 18.0))
    }
    for _, row in df_stops.iloc[1:].iterrows()
]

bench_res = benchmark_runner.run_benchmark(
    start_node=start_node,
    stops_data=stops_data,
    algorithms_to_run=["QPSO", "Simulated Annealing", "Genetic Algorithm", "Ant Colony", "Classical PSO"],
    fleet_size=1,
    traffic_hour=9.0,
    traffic_enabled=True
)

print("\n--- Benchmark Summary Table ---")
bench_res['summary_table']

## 4. Multi-Algorithm Convergence Comparison

In [ ]:
conv_df = bench_res['convergence_df'].set_index('Normalized Progress (%)')

plt.figure(figsize=(10, 5))
for col in conv_df.columns:
    plt.plot(conv_df.index, conv_df[col], label=col, lw=2)

plt.title('Convergence Trajectories Across Algorithms')
plt.xlabel('Optimization Progress (%)')
plt.ylabel('Energy / Distance Minimization')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()